In [1]:
tabla='qtiop10'
col_tabla = "infopefec"
dat= "dat_ceqx002_essi"
col_essi='fec_ope'
essi='essi_dat_cqx002_etl'

In [2]:
from decouple import config
from sqlalchemy import create_engine,text
import pandas as pd
from datetime import datetime, timedelta
import time 
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [5]:
fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

c1= text(query)
connection2.execute(c1)

#INICIO DEL ESSI

In [21]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

query0 = f"""
select
d1.INFOPEORICENASICOD,
d1.INFOPECENASICOD,
d1.INFOPESOLOPENUM,
d1.INFOPESECNUM,
--INFOPEFEC,
to_char(d1.INFOPEFEC, 'YYYY-MM-DD HH24:MI:SS') as INFOPEFEC,
--d1.INFOPEINGSOPHOR,
to_char(d1.INFOPEINGSOPHOR, 'YYYY-MM-DD HH24:MI:SS') as INFOPEINGSOPHOR,
--d1.INFOPESOPTPO,
to_char(d1.INFOPESOPTPO, 'YYYY-MM-DD HH24:MI:SS') as INFOPESOPTPO,
--d1.INFOPEANETPO,
to_char(d1.INFOPEANETPO, 'YYYY-MM-DD HH24:MI:SS') as INFOPEANETPO,
--d1.INFOPEOPETPO,
to_char(d1.INFOPEOPETPO, 'YYYY-MM-DD HH24:MI:SS') as INFOPEOPETPO,
d1.INFOPEEXAPATFLG,
d1.TIPHOPCOD,
d1.INFOPEUSUCRECOD,
--d1.INFOPECREFEC,
to_char(d1.INFOPECREFEC, 'YYYY-MM-DD HH24:MI:SS') as INFOPECREFEC,
INFOPEUSUMODCOD,
--d1.INFOPEMODFEC,
to_char(d1.INFOPEMODFEC, 'YYYY-MM-DD HH24:MI:SS') as INFOPEMODFEC,
d1.DESESOCOD,
d1.INFOPEACTMEDOPENUM,
d1.INFOPEBTB,
d1.INFOPENROGES,
d1.INFOPENROPART,
d1.INFOPECESAANTE,
d1.INFOPETRABPART,
d1.INFOPEEXPUL,
d1.INFOPEDARES,

a1.solopefec as solopefec,  
a1.SOLOPEACTMEDNUM,
a1.SOLOPEBUSPACSECNUM

from {tabla} d1 
left outer join qtsop10 a1 on a1.SOLOPEORICENASICOD = d1.INFOPEORICENASICOD
and a1.SOLOPECENASICOD    = d1.INFOPECENASICOD
and a1.SOLOPENUM    = d1.INFOPESOLOPENUM
where d1.{col_tabla}>='{fecha}'

"""

base1 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [22]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48664 entries, 0 to 48663
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   infopeoricenasicod  48664 non-null  object        
 1   infopecenasicod     48664 non-null  object        
 2   infopesolopenum     48664 non-null  int64         
 3   infopesecnum        48664 non-null  int64         
 4   infopefec           48664 non-null  object        
 5   infopeingsophor     48664 non-null  object        
 6   infopesoptpo        48664 non-null  object        
 7   infopeanetpo        48664 non-null  object        
 8   infopeopetpo        48664 non-null  object        
 9   infopeexapatflg     48664 non-null  object        
 10  tiphopcod           48664 non-null  object        
 11  infopeusucrecod     48664 non-null  object        
 12  infopecrefec        48664 non-null  object        
 13  infopeusumodcod     22344 non-null  object    

In [23]:
base1.columns

Index(['infopeoricenasicod', 'infopecenasicod', 'infopesolopenum',
       'infopesecnum', 'infopefec', 'infopeingsophor', 'infopesoptpo',
       'infopeanetpo', 'infopeopetpo', 'infopeexapatflg', 'tiphopcod',
       'infopeusucrecod', 'infopecrefec', 'infopeusumodcod', 'infopemodfec',
       'desesocod', 'infopeactmedopenum', 'infopebtb', 'infopenroges',
       'infopenropart', 'infopecesaante', 'infopetrabpart', 'infopeexpul',
       'infopedares', 'solopefec', 'solopeactmednum', 'solopebuspacsecnum'],
      dtype='object')

In [25]:
base1 = base1.rename(columns={
    'infopeoricenasicod': 'ori_cas',
    'infopecenasicod': 'cod_cas',
    'infopesolopenum': 'num_sol',
    'infopesecnum': 'num_sec',
    'infopefec': 'fec_ope',
    'infopeingsophor': 'hor_ing_sop',
    'infopesoptpo': 'tie_uso_sop',
    'infopeanetpo': 'tie_ane',
    'infopeopetpo': 'tie_ope',
    'infopeexapatflg': 'res_pat',
    'tiphopcod': 'her_ope',
    'infopeusucrecod': 'usu_cre',
    'infopecrefec': 'fec_cre',
    'infopeusumodcod': 'usu_mod',
    'infopemodfec': 'fec_mod',
    'desesocod': 'cod_des',
    'infopeactmedopenum': 'act_med',
    'infopebtb': 'inf_peb',
    'infopenroges': 'inf_rog',
    'infopenropart': 'inf_per',
    'infopecesaante': 'inf_ces',
    'infopetrabpart': 'inf_tra',
    'infopeexpul': 'inf_exp',
    'infopedares': 'inf_are',
    'solopefec': 'sol_fec',
    'solopeactmednum': 'sol_act',
    'solopebuspacsecnum': 'pac_sec'
})

In [26]:
base1.shape

(48664, 27)

In [27]:
base2=pd.read_sql_query(f"SELECT * FROM {essi} LIMIT 10", con=connection1)

In [28]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 33 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ori_cas      0 non-null      object
 1   cod_cas      0 non-null      object
 2   des_cas      0 non-null      object
 3   cod_red      0 non-null      object
 4   des_red      0 non-null      object
 5   num_sol      0 non-null      object
 6   num_sec      0 non-null      object
 7   fec_ope      0 non-null      object
 8   hor_ing_sop  0 non-null      object
 9   tie_uso_sop  0 non-null      object
 10  tie_ane      0 non-null      object
 11  tie_ope      0 non-null      object
 12  res_pat      0 non-null      object
 13  des_res      0 non-null      object
 14  her_ope      0 non-null      object
 15  des_her      0 non-null      object
 16  usu_cre      0 non-null      object
 17  fec_cre      0 non-null      object
 18  usu_mod      0 non-null      object
 19  fec_mod      0 non-null      object
 20  cod_d

#TRAEMOS TODOS LOS MAESTROS

In [29]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
#id_red,cod_cas,des_cas,cod_red,des_red

cqxrespat=pd.read_sql_query(f"SELECT cod_res,des_res FROM dim_cqxrespat", con=connection2)
cqxrespat['cod_res']=cqxrespat['cod_res'].str.strip()

cqxtipher=pd.read_sql_query(f"SELECT cod_the,des_the FROM dim_cqxtipher", con=connection2)
cqxtipher['cod_the']=cqxtipher['cod_the'].str.strip()

cqxdesegrsop=pd.read_sql_query(f"SELECT des_cod,des_des FROM dim_cqxdesegrsop", con=connection2)
cqxdesegrsop['des_cod']=cqxdesegrsop['des_cod'].str.strip()

In [30]:
a=base1.copy()

In [33]:
base1=a

In [31]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48664 entries, 0 to 48663
Data columns (total 27 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ori_cas      48664 non-null  object        
 1   cod_cas      48664 non-null  object        
 2   num_sol      48664 non-null  int64         
 3   num_sec      48664 non-null  int64         
 4   fec_ope      48664 non-null  object        
 5   hor_ing_sop  48664 non-null  object        
 6   tie_uso_sop  48664 non-null  object        
 7   tie_ane      48664 non-null  object        
 8   tie_ope      48664 non-null  object        
 9   res_pat      48664 non-null  object        
 10  her_ope      48664 non-null  object        
 11  usu_cre      48664 non-null  object        
 12  fec_cre      48664 non-null  object        
 13  usu_mod      22344 non-null  object        
 14  fec_mod      22344 non-null  object        
 15  cod_des      48664 non-null  object        
 16  act_

In [32]:
base1=pd.merge(base1,cas_red,how='left',on='cod_cas')
base1=base1.drop("id_red", axis=1)
base1.shape

(48664, 30)

In [33]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 48664 entries, 0 to 48663
Data columns (total 30 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ori_cas      48664 non-null  object        
 1   cod_cas      48664 non-null  object        
 2   num_sol      48664 non-null  int64         
 3   num_sec      48664 non-null  int64         
 4   fec_ope      48664 non-null  object        
 5   hor_ing_sop  48664 non-null  object        
 6   tie_uso_sop  48664 non-null  object        
 7   tie_ane      48664 non-null  object        
 8   tie_ope      48664 non-null  object        
 9   res_pat      48664 non-null  object        
 10  her_ope      48664 non-null  object        
 11  usu_cre      48664 non-null  object        
 12  fec_cre      48664 non-null  object        
 13  usu_mod      22344 non-null  object        
 14  fec_mod      22344 non-null  object        
 15  cod_des      48664 non-null  object        
 16  act_

In [34]:
#des_res
base1['res_pat']=base1['res_pat'].str.strip()
base1=pd.merge(base1,cqxrespat,how='left',left_on='res_pat',right_on='cod_res')
base1 = base1.drop("cod_res", axis=1)
base1.shape

(48664, 31)

In [35]:
#des_her
base1['her_ope']=base1['her_ope'].str.strip()
base1=pd.merge(base1,cqxtipher,how='left',left_on='her_ope',right_on='cod_the')
base1=base1.rename(columns={"des_the":"des_her"})
base1 = base1.drop("cod_the", axis=1)
base1.shape

(48664, 32)

In [36]:
#des_des
base1['cod_des']=base1['cod_des'].str.strip()
base1=pd.merge(base1,cqxdesegrsop,how='left',left_on='cod_des',right_on='des_cod')
base1 = base1.drop("des_cod", axis=1)
base1.shape 

(48664, 33)

In [37]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 48664 entries, 0 to 48663
Data columns (total 33 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ori_cas      48664 non-null  object        
 1   cod_cas      48664 non-null  object        
 2   num_sol      48664 non-null  int64         
 3   num_sec      48664 non-null  int64         
 4   fec_ope      48664 non-null  object        
 5   hor_ing_sop  48664 non-null  object        
 6   tie_uso_sop  48664 non-null  object        
 7   tie_ane      48664 non-null  object        
 8   tie_ope      48664 non-null  object        
 9   res_pat      48664 non-null  object        
 10  her_ope      48664 non-null  object        
 11  usu_cre      48664 non-null  object        
 12  fec_cre      48664 non-null  object        
 13  usu_mod      22344 non-null  object        
 14  fec_mod      22344 non-null  object        
 15  cod_des      48664 non-null  object        
 16  act_

In [38]:
base2.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'num_sol',
       'num_sec', 'fec_ope', 'hor_ing_sop', 'tie_uso_sop', 'tie_ane',
       'tie_ope', 'res_pat', 'des_res', 'her_ope', 'des_her', 'usu_cre',
       'fec_cre', 'usu_mod', 'fec_mod', 'cod_des', 'des_des', 'act_med',
       'inf_peb', 'inf_rog', 'inf_per', 'inf_ces', 'inf_tra', 'inf_exp',
       'inf_are', 'sol_fec', 'sol_act', 'pac_sec'],
      dtype='object')

In [39]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [40]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [41]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [68]:
base.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 48664 entries, 0 to 48663
Data columns (total 33 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ori_cas      48664 non-null  object        
 1   cod_cas      48664 non-null  object        
 2   inf_peb      48661 non-null  float64       
 3   num_sol      48664 non-null  int64         
 4   sol_act      48664 non-null  int64         
 5   inf_are      48661 non-null  float64       
 6   inf_per      48661 non-null  float64       
 7   usu_cre      48664 non-null  object        
 8   cod_red      48664 non-null  object        
 9   fec_ope      48664 non-null  object        
 10  fec_mod      22344 non-null  object        
 11  des_her      48664 non-null  object        
 12  usu_mod      22344 non-null  object        
 13  inf_exp      48661 non-null  float64       
 14  des_res      48664 non-null  object        
 15  inf_tra      48661 non-null  float64       
 16  hor_

In [42]:
base.head()

,ori_cas,cod_cas,inf_peb,num_sol,sol_act,inf_are,inf_per,usu_cre,cod_red,fec_ope,...,act_med,tie_uso_sop,inf_rog,cod_des,tie_ane,tie_ope,her_ope,num_sec,fec_cre,pac_sec
0,1,074,0.0,107,7509,0.0,0.0,41926350,18,2023-06-01 00:00:00,...,843091,0001-01-01 08:00:00,0.0,09,0001-01-01 01:00:00,0001-01-01 01:30:00,1,1,2023-06-01 09:21:09,8945335
1,1,162,0.0,520,21708,0.0,0.0,41841095,17,2023-06-16 00:00:00,...,1417545,0001-01-01 11:00:00,0.0,00,0001-01-01 00:00:00,0001-01-01 00:00:00,1,1,2023-06-16 10:12:01,9269304
2,1,139,0.0,2389,160854,0.0,0.0,29590473,34,2023-06-12 00:00:00,...,1820940,0001-01-01 08:15:00,0.0,09,0001-01-01 00:00:00,0001-01-01 00:00:00,1,1,2023-06-12 08:12:36,25838981
3,1,003,0.0,274,2250558,0.0,0.0,44719956,18,2023-05-30 00:00:00,...,2251697,0001-01-01 15:45:00,0.0,01,0001-01-01 01:20:00,0001-01-01 01:30:00,1,1,2023-05-30 15:49:43,23095886
4,1,102,0.0,2473,161531,0.0,0.0,08584398,30,2023-05-26 00:00:00,...,1412603,0001-01-01 09:30:00,0.0,09,0001-01-01 01:20:00,0001-01-01 02:50:00,1,1,2023-05-26 08:41:32,1137529


In [43]:
base.columns

Index(['ori_cas', 'cod_cas', 'inf_peb', 'num_sol', 'sol_act', 'inf_are',
       'inf_per', 'usu_cre', 'cod_red', 'fec_ope', 'fec_mod', 'des_her',
       'usu_mod', 'inf_exp', 'des_res', 'inf_tra', 'hor_ing_sop', 'res_pat',
       'des_red', 'inf_ces', 'sol_fec', 'des_cas', 'des_des', 'act_med',
       'tie_uso_sop', 'inf_rog', 'cod_des', 'tie_ane', 'tie_ope', 'her_ope',
       'num_sec', 'fec_cre', 'pac_sec'],
      dtype='object')

In [45]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [73]:
base.to_sql(name=f'{essi}', con=connection1, if_exists='append', index=False,chunksize=10000)

4664

In [74]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3)
print("Proceso 1.2 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.2 completado satisfactoriamente en 2677.924 segundos


In [ ]:
connection1.close()
connection2.close()
connection3.close()

In [ ]:
engine1.dispose()
engine2.dispose()
engine3.dispose()